# 03 · Drift, adaptación por olvido y decisión

Qué se responde aquí:

1. ¿Compensa olvidar? ¿Qué tamaño de ventana?
2. ¿Los detectores encuentran el cambio, y con cuánto retraso?
3. ¿Qué **no** consigue el sistema?

Este es el experimento central: cinco estrategias sobre **los mismos eventos**, la
misma semilla, la misma política y la misma familia.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent          # el notebook puede abrirse desde notebooks/
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fraud_adaptive.pipeline import load_configs
from fraud_adaptive.tracking import read_json

CONFIGS = load_configs(ROOT / "configs")
RUN_DIR = ROOT / CONFIGS["base"]["paths"]["runs"] / "principal"
REPORTS = ROOT / CONFIGS["base"]["paths"]["reports"]

MANIFEST = read_json(ROOT / CONFIGS["base"]["paths"]["manifests"] / "sources.json")
DATA_SOURCE = MANIFEST.get("data_source", "desconocido")

if DATA_SOURCE == "sintetico_sustituto":
    print("AVISO: los datos son un SUSTITUTO SINTETICO. Ninguna cifra describe IEEE-CIS.")
else:
    print("Datos: IEEE-CIS Fraud Detection (particion train).")
print("Run:", RUN_DIR)

## 1. El protocolo temporal

Las tres reservas de 7 días son idénticas para todas las estrategias. Eso es lo que hace que una diferencia de costo sea atribuible al **tamaño de ventana** y no a otra cosa.

In [ ]:
splits = read_json(ROOT / CONFIGS["base"]["paths"]["manifests"] / "splits.json")

print(f"L = {splits['label_delay_days']} días   cadencia = {splits['cadence_days']} días")
print(f"Regla de elegibilidad: {splits['eligibility_rule']}\n")

filas = []
for estrategia, versiones in splits["versions"].items():
    for v in versiones:
        filas.append({
            "version": v["version_id"], "T": v["update_time"], "c": v["cutoff"],
            "predictor": f"[{v['predictor'][0]},{v['predictor'][1]})",
            "dias_fit": v["predictor_days"],
            "calibracion": f"[{v['calibration'][0]},{v['calibration'][1]})",
            "H": f"[{v['promotion_validation'][0]},{v['promotion_validation'][1]})",
        })
pd.DataFrame(filas)

## 2. Resultado central

Una fila por estrategia, con las tres familias de métricas. `S0` y `E15` son **referencias no desplegables**.

In [ ]:
adaptacion = pd.read_csv(REPORTS / "tables" / "adaptacion.csv")
adaptacion[["estrategia", "ap", "skill", "brier", "f1", "costo_um_tx",
            "costo_ic_low", "costo_ic_high", "bloqueo_legitimas",
            "n_revisiones", "versiones_activadas"]].round(4)

### Degradación por bloque

Aquí se ve si el olvido sirve: una ventana corta debería sostener el AP mientras el estático cae.

In [ ]:
adaptacion[["estrategia", "ap_B1", "ap_B2", "ap_B3", "ap_B4"]].round(4)

### Diferencia pareada frente a las referencias

Remuestreo de bloques contiguos de 7 días sobre los **mismos eventos**. Un intervalo que no cruza el cero indica una diferencia consistente.

In [ ]:
cols = [c for c in adaptacion.columns if c.startswith("delta_costo_vs_")]
adaptacion[["estrategia"] + cols].round(4)

## 3. ¿Se cumplen los objetivos declarados?

Un objetivo **puede incumplirse**. Se reporta el nivel realmente obtenido, no el objetivo.

In [ ]:
objetivos = CONFIGS["decision"]["diagnostic_targets"]
tabla = adaptacion[["estrategia", "recall_at_fpr1", "fpr_obtenido",
                    "recall_at_precision80", "precision_obtenida"]].round(4)
print(f"Objetivos: FPR <= {objetivos['fpr_target']}, precisión >= {objetivos['precision_target']}\n")
display(tabla)

print("\nFPR objetivo alcanzado en test:",
      bool((tabla['fpr_obtenido'] <= objetivos['fpr_target']).any()))
print("Precisión objetivo alcanzada en test:",
      bool((tabla['precision_obtenida'] >= objetivos['precision_target']).any()))
print("\nLos umbrales se eligieron en validación y se aplicaron CONGELADOS.")
print("Que no se sostengan en test es un resultado, no un error a corregir moviendo el umbral.")

## 4. Señales de drift y sus dos relojes

KS/PSI y S1 están disponibles el mismo día. ADWIN observa el **error real** y por eso llega L=30 días después.

In [ ]:
drift_path = RUN_DIR / "drift_log.csv"
if drift_path.exists():
    drift = pd.read_csv(drift_path)
    alertas = drift[drift["alerta"].fillna(False).astype(bool)] if "alerta" in drift else pd.DataFrame()
    print(f"Registros de señal: {len(drift)}   alertas: {len(alertas)}")
    if not alertas.empty:
        display(alertas.groupby("senal").agg(
            n_alertas=("senal", "size"),
            primer_dia=("dia_evento", "min"),
            retraso_medio=("retraso_dias", "mean"),
        ).round(2))
else:
    print("Sin bitácora de drift en este run.")

In [ ]:
adwin_path = RUN_DIR / "adwin_detections.csv"
if adwin_path.exists():
    adwin = pd.read_csv(adwin_path)
    print(f"Detecciones de ADWIN: {len(adwin)}")
    display(adwin[["estrategia", "version_id", "dia_evento", "dia_disponibilidad",
                   "retraso_dias", "n_updates"]])
else:
    print("ADWIN no detectó cambios en ninguna versión.")

### El detector, probado contra verdad conocida

Validación del **instrumento**: no son resultados sobre fraude.

In [ ]:
pd.read_csv(REPORTS / "tables" / "adwin_selftest.csv")

## 5. Sensibilidad económica

Rescoring de predicciones guardadas con la política **congelada**. Responde *cuánto cambiaría la conclusión*, no *qué política sería óptima*.

In [ ]:
sens_path = REPORTS / "tables" / "sensibilidad_economica.csv"
if sens_path.exists():
    display(pd.read_csv(sens_path).round(4))
else:
    print("Sensibilidad no disponible en este run.")

## 6. Equidad operativa

Segmentos de **negocio**. IEEE-CIS no tiene atributos protegidos verificables: esto no es una auditoría demográfica.

In [ ]:
seg_path = REPORTS / "tables" / "segmentos.csv"
if seg_path.exists():
    seg = pd.read_csv(seg_path)
    display(seg[seg["soporte_suficiente"]][
        ["variable", "grupo", "n_legitimas", "tasa_bloqueo", "brecha_pp", "alerta_brecha"]
    ].round(4))
    print(f"\nGrupos con evidencia insuficiente: {int(seg['evidencia_insuficiente'].sum())}")
    print("Se conservan en la tabla: ocultarlos daría una falsa impresión de cobertura.")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(REPORTS / "figures" / "drift_adaptacion.png")))

## 7. Costo de cómputo

In [ ]:
display(pd.read_csv(REPORTS / "tables" / "costos_computo.csv").round(2))

presupuesto = read_json(RUN_DIR / "budget.json")
print(f"\nConsumido: {presupuesto['consumed_minutes']:.1f} min de "
      f"{presupuesto['total_minutes']:.0f}  ({presupuesto['remaining_minutes']:.1f} restantes)")

## 8. Qué no demuestra este experimento

- No es una validación prospectiva: es un backtest retrospectivo, y la selección
  previa del dataset ya examinó periodos tardíos.
- Los costos son simulados bajo supuestos declarados; no hay ahorro causal medido.
- Una sola semilla no permite afirmar variabilidad entre semillas.
- La ventana recomendada es la elegida **en desarrollo**. Si otra resulta mejor en
  el test final, eso es un diagnóstico retrospectivo, no un permiso para retunear.